# Candle Lab: fine-tune Kronos-small on crypto 15-minute candles (Kaggle, free)

This notebook fine-tunes **Kronos-small** with Kronos's own CSV fine-tuning code (`finetune_csv/`), using
free Coinbase candles for BTC, ETH and SOL. It outputs a small model folder that Candle Lab's daily
experiment can test as a **challenger**. It never touches money or exchange accounts.

## How to run it (step by step)
1. Sign in at [kaggle.com](https://www.kaggle.com) (free). The first time, verify your phone number
   (Settings → Phone verification). Kaggle needs this before it allows internet or GPUs.
2. **Create → New Notebook → File → Import Notebook**, and upload this `.ipynb` file.
3. In the right-hand panel, under **Session options**:
   * **Accelerator** → `GPU T4 x2` (or `GPU P100`). The free quota is about 30 GPU hours a week, and this needs about 1 hour.
   * **Internet** → **On**.
4. Click **Run All**. Leave the tab open. It downloads data, fine-tunes on BTC, then ETH, then SOL.
5. When it finishes, open the **Output** panel (right side, `/kaggle/working`) and download
   **`kronos-crypto-ft.zip`**.
6. In your Candle Lab repo, unzip it to `models/kronos-crypto-ft/`, so the folder contains
   `config.json` and `model.safetensors`. Commit and push. At ~50 MB it is under GitHub's 100 MB file limit.
   (On github.com: open the repo, **Add file → Upload files**, then drag in the two files and set the path
   to `models/kronos-crypto-ft/`.)
7. Edit `config/experiment.json` and set:
   ```json
   "extra_models": [{"model": "models/kronos-crypto-ft", "tokenizer": "NeoQuasar/Kronos-Tokenizer-base", "max_context": 512}]
   ```
   From the next daily run on, one challenger each day uses the fine-tuned model, with the champion's other settings.

### Important: what data it trains on
By default it trains on **2024-07-01 → 2025-12-31**, which is the *tuning* split. So the fine-tuned model's **tuning**
score is in-sample (it has seen those candles) and will look too good. That is why promotion also requires
beating the champion on the **locked holdout** (2026) and surviving **2 weeks of live probation**. Never move
`TRAIN_END` past the start of the holdout split, or the holdout check stops meaning anything.

In [ ]:
# ---- Settings (edit if you like) ----
COINS = ["BTC-USD", "ETH-USD", "SOL-USD"]   # trained one after another, each continuing from the last
TRAIN_START = "2024-07-01"                  # after Kronos's pre-training cutoff (June 2024)
TRAIN_END = "2025-12-31"                    # must be BEFORE the holdout split in config/experiment.json
LOOKBACK = 256                              # context length (Candle Lab's default champion uses 256)
PREDICT = 16                                # future candles per training example
EPOCHS_PER_COIN = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5                        # Kronos's template default is 1e-6 (very gentle)
KRONOS_COMMIT = "67b630e67f6a18c9e9be918d9b4337c960db1e9a"   # the version Candle Lab vendors

In [ ]:
import subprocess, sys, os
if not os.path.exists("Kronos"):
    subprocess.run(["git", "clone", "https://github.com/shiyu-coder/Kronos.git"], check=True)
subprocess.run(["git", "-C", "Kronos", "checkout", "-q", KRONOS_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "safetensors", "huggingface_hub", "pyyaml"], check=True)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on a GPU accelerator!")

In [ ]:
# Download 15-minute candles from Coinbase's free public API (300 per request).
import time, requests, pandas as pd
from datetime import datetime, timezone

def iso(ts): return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def fetch(product, start, end):
    s = int(pd.Timestamp(start, tz="UTC").timestamp()); e = int(pd.Timestamp(end, tz="UTC").timestamp()) + 86400 - 900
    rows, cur = [], s
    while cur <= e:
        ce = min(cur + 299 * 900, e)
        for attempt in range(5):
            r = requests.get(f"https://api.exchange.coinbase.com/products/{product}/candles",
                             params={"granularity": 900, "start": iso(cur), "end": iso(ce)},
                             headers={"User-Agent": "candle-lab-finetune"}, timeout=20)
            if r.status_code == 200: break
            time.sleep(2 ** attempt)
        r.raise_for_status()
        rows += [(x[0], x[3], x[2], x[1], x[4]) for x in r.json()]   # time, open, high, low, close
        cur = ce + 900
        time.sleep(0.35)
    df = pd.DataFrame(rows, columns=["ts", "open", "high", "low", "close"]).drop_duplicates("ts").sort_values("ts")
    # Fill missing 15-minute slots with flat candles (same as Candle Lab does)
    df = df.set_index("ts").reindex(range(df.ts.iloc[0], df.ts.iloc[-1] + 900, 900))
    df["close"] = df["close"].ffill()
    for c in ["open", "high", "low"]: df[c] = df[c].fillna(df["close"])
    df = df.reset_index().rename(columns={"index": "ts"})
    df["timestamps"] = pd.to_datetime(df["ts"], unit="s")
    # Candle Lab feeds Kronos no volume for crypto (as in the Kronos paper), so train the same way.
    df["volume"] = 0.0; df["amount"] = 0.0
    return df[["timestamps", "open", "high", "low", "close", "volume", "amount"]]

os.makedirs("data", exist_ok=True)
for coin in COINS:
    path = f"data/{coin}.csv"
    if not os.path.exists(path):
        print("downloading", coin, "...")
        fetch(coin, TRAIN_START, TRAIN_END).to_csv(path, index=False)
    d = pd.read_csv(path); print(coin, len(d), "candles", d.timestamps.iloc[0], "->", d.timestamps.iloc[-1])

In [ ]:
# Base weights from Hugging Face (public, no key needed).
from huggingface_hub import snapshot_download
TOKENIZER_DIR = snapshot_download("NeoQuasar/Kronos-Tokenizer-base")
BASE_MODEL_DIR = snapshot_download("NeoQuasar/Kronos-small")
print(TOKENIZER_DIR, BASE_MODEL_DIR)

In [ ]:
# Fine-tune the predictor on each coin in turn, using Kronos's finetune_csv/train_sequential.py.
# The tokenizer is NOT retrained: Candle Lab pairs the result with NeoQuasar/Kronos-Tokenizer-base.
import yaml
template = yaml.safe_load(open("Kronos/finetune_csv/configs/config_ali09988_candle-5min.yaml"))
current = BASE_MODEL_DIR
WORK = os.path.abspath("finetuned")
for coin in COINS:
    cfg = yaml.safe_load(yaml.safe_dump(template))
    cfg["data"].update(data_path=os.path.abspath(f"data/{coin}.csv"), lookback_window=LOOKBACK,
                       predict_window=PREDICT, max_context=512, train_ratio=0.9, val_ratio=0.1, test_ratio=0.0)
    cfg["training"].update(basemodel_epochs=EPOCHS_PER_COIN, batch_size=BATCH_SIZE,
                           predictor_learning_rate=LEARNING_RATE, num_workers=2)
    exp = f"crypto_{coin.replace('-', '_')}"
    cfg["model_paths"].update(pretrained_tokenizer=TOKENIZER_DIR, pretrained_predictor=current, exp_name=exp,
                              base_path=WORK, base_save_path="", finetuned_tokenizer=TOKENIZER_DIR)
    cfg["experiment"].update(use_comet=False, train_tokenizer=False, train_basemodel=True, skip_existing=False)
    cfg["device"] = {"use_cuda": True, "device_id": 0}
    cfg_path = os.path.abspath(f"config_{exp}.yaml")
    yaml.safe_dump(cfg, open(cfg_path, "w"))
    print(f"\n===== fine-tuning on {coin} (starting from {current}) =====", flush=True)
    r = subprocess.run([sys.executable, "train_sequential.py", "--config", cfg_path, "--skip-tokenizer"],
                       cwd="Kronos/finetune_csv")
    if r.returncode != 0:
        raise RuntimeError(f"training failed on {coin}; scroll up for the error")
    current = os.path.join(WORK, exp, "basemodel", "best_model")
    print("saved:", current, os.listdir(current))
FINAL = current

In [ ]:
# Package for Candle Lab: fp16 weights (~50 MB, under GitHub's 100 MB file limit) + config.json.
import json, shutil, zipfile
from safetensors.torch import load_file, save_file
out = "/kaggle/working/kronos-crypto-ft" if os.path.exists("/kaggle/working") else "kronos-crypto-ft"
os.makedirs(out, exist_ok=True)
weights = load_file(os.path.join(FINAL, "model.safetensors"))
save_file({k: v.half().contiguous() for k, v in weights.items()}, os.path.join(out, "model.safetensors"))
shutil.copy(os.path.join(FINAL, "config.json"), os.path.join(out, "config.json"))
json.dump({"base": "NeoQuasar/Kronos-small", "tokenizer": "NeoQuasar/Kronos-Tokenizer-base", "coins": COINS,
           "train_start": TRAIN_START, "train_end": TRAIN_END, "lookback": LOOKBACK, "epochs_per_coin": EPOCHS_PER_COIN,
           "learning_rate": LEARNING_RATE, "kronos_commit": KRONOS_COMMIT},
          open(os.path.join(out, "TRAINING_INFO.json"), "w"), indent=2)
with zipfile.ZipFile(out + ".zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(out): z.write(os.path.join(out, f), f)
print("Done. Download", out + ".zip", "from the Output panel.")
for f in os.listdir(out): print(f, round(os.path.getsize(os.path.join(out, f)) / 1e6, 1), "MB")

In [ ]:
# Sanity check: the packaged model loads and gives a forecast.
sys.path.insert(0, "Kronos")
from model import Kronos, KronosTokenizer, KronosPredictor
m = Kronos.from_pretrained(out); t = KronosTokenizer.from_pretrained(TOKENIZER_DIR)
p = KronosPredictor(m.float(), t, max_context=512)
d = pd.read_csv(f"data/{COINS[0]}.csv"); d["timestamps"] = pd.to_datetime(d["timestamps"])
x = d.iloc[-LOOKBACK - 1:-1]
pred = p.predict(x[["open", "high", "low", "close"]], x["timestamps"].reset_index(drop=True),
                 d["timestamps"].iloc[-1:].reset_index(drop=True), pred_len=1, sample_count=5, verbose=False)
print("last close", x.close.iloc[-1], "-> predicted", round(pred.close.iloc[0], 2), "| actual", d.close.iloc[-1])

## Optional: share the model on Hugging Face instead of committing it
If you have a free Hugging Face account, add a **write** token as a Kaggle secret named `HF_TOKEN`
(Add-ons → Secrets), then run the cell below with your own repo name. In `config/experiment.json`, use
`"model": "your-username/kronos-crypto-ft"` instead of the local folder.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import HfApi
# token = UserSecretsClient().get_secret("HF_TOKEN")
# HfApi(token=token).create_repo("your-username/kronos-crypto-ft", exist_ok=True)
# HfApi(token=token).upload_folder(folder_path=out, repo_id="your-username/kronos-crypto-ft")